## Setup

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.status()

In [ ]:
using CairoMakie
using Carlo.ResultTools
using CarloAnalysis
using DataFrames
using FFTW
using HDF5
using JLD2
using LinearAlgebra
using StaticArrays
using Statistics
using WignerMolecule

set_theme!(theme_latexfonts())

In [ ]:
# Boltzmann constant in meV/K
const kB = 8.617333262e-2

In [ ]:
getetapar(ηk) = real(ηk[1,1] + ηk[2,2])
getetapar(ηk::Vector) = getetapar(reshape(ηk, (3,3)))

In [ ]:
function generate_spins(jobname, task_no)
    fig = Figure(size=(800, 400))

    task_str = lpad(task_no, 4, "0")
    h5open("../jobs/$jobname.data/task$task_str/run0001.dump.h5") do file
        spins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spin_xs = map(v -> v[1], spins)
        spin_ys = map(v -> v[2], spins)
        spin_zs = map(v -> v[3], spins)
        Lx, Ly = size(spins)
        fig[1,1] = Axis(fig; title="Spins", backgroundcolor="black")
        strength = vec(spin_zs)
        arrows2d!(1:Lx, 1:Ly, spin_xs, spin_ys, lengthscale=0.5, align=:center, color=strength,
                  colorrange=(-1, 1))

        ηs = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        η_xs = getindex.(ηs, 1)
        η_ys = getindex.(ηs, 2)
        η_zs = getindex.(ηs, 3)
        Lx, Ly = size(ηs)
        fig[1,2] = Axis(fig; title="ηs", backgroundcolor="black")
        strength = vec(η_zs)
        arrows2d!(1:Lx, 1:Ly, η_xs, η_ys, lengthscale=0.5, align=:center, color=strength,
                  colorrange=(-1, 1))
    end

    return fig
end

In [ ]:
function generate_spinks(jobname, task_no; run_no=1)
    fig = Figure(size=(800, 500))

    task_str = lpad(task_no, 4, "0")
    run_str = lpad(run_no, 4, "0")
    h5open("../jobs/$jobname.data/task$task_str/run$run_str.dump.h5") do file
        rawetas = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        etas = zeros(size(rawetas)..., 3)
        for I in eachindex(IndexCartesian(), rawetas)
            etas[I, :] = rawetas[I]
        end
        etaks = fft(etas, (1, 2)) / length(rawetas)
        spin_mags = sum(abs2.(etaks), dims=3)[:,:,1]
        fig[1,1] = ax = Axis(fig; title="ηk correlations")
        hm = heatmap!(ax, spin_mags)
        Colorbar(fig[2,1], hm, vertical=false, flipaxis=false)

        rawspins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spins = zeros(size(rawspins)..., 3)
        for I in eachindex(IndexCartesian(), rawspins)
            spins[I, :] = rawspins[I]
        end
        spinks = fft(spins, (1, 2)) / length(rawspins)
        spin_mags = sum(abs2.(spinks), dims=3)[:,:,1]
        fig[1,2] = ax = Axis(fig; title="sk correlations")
        hm = heatmap!(ax, spin_mags)
        Colorbar(fig[2,2], hm, vertical=false, flipaxis=false)
    end

    return fig
end

In [ ]:
function corrlen(avgcorrs; rtol=0.35)
    avgcorrs = log.(abs.(avgcorrs))
    slopes = [avgcorrs[i+1] - avgcorrs[i] for i in 1:(length(avgcorrs)-1)]
    i = 1
    while true
        if i == length(slopes)
            throw(ErrorException("Could not find noise threshold"))
        end
        slope = sum(slopes[1:i]) / i
        newslope = slopes[i+1]
        i += 1
        if abs((newslope - slope) / slope) > rtol
            break
        end
    end
    noise = sum(avgcorrs[i+1:end]) / (length(avgcorrs) - i)
    xis = [(noise - avgcorrs[j]) / slopes[j] + (j-1) for j in 1:(i-1)]

    return sum(xis) / length(xis)
end

In [ ]:
checkeq(x) = y -> y .== x
selectamer(df, am, er) = subset(df, :am => checkeq(am), :er => checkeq(er))
function getscorr(df, am, er, phase)
    df = selectamer(df, am, er)
    if phase == :stripe
        return sum(df[1, [:sk_corr_half_M, :sk_corr_half_M2, :sk_corr_half_M3]])
    elseif phase == :fm
        return df[1, :sk_corr_Γ]
    elseif phase == :afm_fe
        return sum(df[1, [:sk_corr_M, :sk_corr_M2, :sk_corr_M3]])
    elseif phase == :afm_afe
        return sum(df[1, [:sk_corr_part_K, :sk_corr_part_K2, :sk_corr_part_K3]])
    end
end
function getηcorr(df, am, er, phase)
    df = selectamer(df, am, er)
    return df[1, Symbol("etak_corr_$phase")]
end

In [ ]:
function linfit(xs, ys; weight=true)
    A = ones(length(xs), 2)
    A[:, 2] = xs
    if weight
        W = Diagonal(getfield.(ys, :err) .^ (-2))
    else
        W = Diagonal(ones(length(ys)))
    end
    return (A' * W * A) \ (A' * W * ys)
end

# Extrapolate to infinite system size
function extrapolate(Ls, ys)
    β = linfit(Ls .^ (-2), ys, weight=true)
    return β[1]
end
pearson(Ls, ys) = cor(Ls .^ (-2), ys)

In [ ]:
function estTc(Ts, ords; thresh=0.1)
    ids = filter(i -> ords[i] >= thresh, 1:length(Ts))
    return length(ids) == 0 ? 0 : maximum([Ts[i] for i in ids])
end

In [ ]:
getH0(am, er) = norm(load_object("params_interpolated_quarter.jld2")[(am, er)])

In [ ]:
inch = 96
pt = 4/3
cm = inch / 2.54
width = 8.5cm

## er = 7

In [ ]:
results = JobResult("../jobs", "er7")
df = results.data
df[:, :XiSpin] = corrlen.(df[:, :sr_corr])
df[:, :XiEta] = corrlen.(df[:,:etar_corr])
for phase in [:stripe, :afm_afe, :afm_fe]
    df[:, Symbol(phase, "_auto")] = get_autocorr(results, Symbol("etak_corr_", phase))
end
Ts = unique(df[:,:T])
results

In [ ]:
fig = Figure(size=(1000, 600))

setL = 48
df = groupby(subset(results.data, :Lx => L -> L .== setL), :T)
for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{Order} (T=%$T)", xlabel=L"a_m")
    data = df[(T=T,)]
    generate_plot!(ax, :am, :etak_corr_stripe, data; line=true, label="stripe")
    generate_plot!(ax, :am, :etak_corr_afm_fe, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :etak_corr_afm_afe, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 600))

setL = 48
df = groupby(subset(results.data, :Lx => L -> L .== setL), :T)
for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\rho$ \textbf{Kurtosis} (T=%$T)", xlabel=L"a_m")
    data = df[(T=T,)]
    generate_plot!(ax, :am, :rho_kurt_stripe, data; line=true, label="stripe")
    generate_plot!(ax, :am, :rho_kurt_afm_fe, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :rho_kurt_afm_afe, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 600))

setL = 48
df = groupby(subset(results.data, :Lx => L -> L .== setL), :T)
for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{Kurtosis} (T=%$T)", xlabel=L"a_m")
    data = df[(T=T,)]
    generate_plot!(ax, :am, :etak_kurt_stripe, data; line=true, label="stripe")
    generate_plot!(ax, :am, :etak_kurt_afm_fe, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :etak_kurt_afm_afe, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 600))

setL = 48
df = groupby(subset(results.data, :Lx => L -> L .== setL), :T)
for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$S$ \textbf{Order} (T=%$T)", xlabel=L"r")
    data = df[(T=T,)]
    generate_plot!(ax, :am, :sk_corr_half_M, data; line=true, label="stripe")
    generate_plot!(ax, :am, :sk_corr_M, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :sk_corr_part_K, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 600))

setam = 8.5
setL = 48
df = groupby(subset(results.data, :Lx => L -> L .== setL, :am => am -> am .== setam), :T)
for i in eachindex(Ts)
    T = Ts[i]
    ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{real corrs} (T=%$T)", xlabel=L"r")
    data = df[(T=T,)]
    rs = collect(0:(div(setL, 2) - 1))
    rawcorrs = data[1, :etar_corr]
    rcorrs = log10.(abs.(rawcorrs[:,1] + rawcorrs[:,2]))
    generate_plot!(ax, rs, rcorrs; line=true, label="stripe")
    mincorr = minimum(getfield.(rcorrs, :val))
    maxcorr = maximum(getfield.(rcorrs, :val))
    ylims!(low=1.02*mincorr, high=0.98*maxcorr)
end

fig

In [ ]:
fig = Figure(size=(900, 600))

setL = 48
setT = 0.5
ax1 = fig[1,1] = Axis(fig, title=L"\textbf{Eta Corr Len vs.} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :XiEta, df; line=true, label="stripe")

setT = 1.0
ax2 = fig[2,1] = Axis(fig, title=L"\textbf{Eta Corr Len vs.} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :XiEta, df; line=true, label="stripe")

setT = 1.5
ax1 = fig[1,2] = Axis(fig, title=L"\textbf{Eta Corr Len vs.} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :XiEta, df; line=true, label="stripe")

setT = 2.0
ax2 = fig[2,2] = Axis(fig, title=L"\textbf{Eta Corr Len vs.} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :XiEta, df; line=true, label="stripe")

setT = 3.0
ax1 = fig[1,3] = Axis(fig, title=L"\textbf{Eta Corr Len vs.} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :XiEta, df; line=true, label="stripe")

setT = 4.0
ax2 = fig[2,3] = Axis(fig, title=L"\textbf{Eta Corr Len vs.} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :XiEta, df; line=true, label="stripe")

fig

In [ ]:
fig = Figure(size=(1000, 600))

setL = 48
df = groupby(subset(results.data, :Lx => L -> L .== setL), :T)
for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{Autocorrelation} (T=%$T)", xlabel=L"a_m")
    data = df[(T=T,)]
    generate_plot!(ax, :am, :stripe_auto, data; line=true, label="stripe")
    generate_plot!(ax, :am, :afm_fe_auto, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :afm_afe_auto, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(800, 400))

df = results.data
ax1 = fig[1,1] = Axis(fig, title="Energy vs. am", xlabel=L"a_m")
generate_plot!(ax1, :am, :Energy, [:T], df; line=true)
ax2 = fig[1,2] = Axis(fig, title="Heat Capacity vs. am", xlabel=L"a_m")
generate_plot!(ax2, :am, :HeatCap, [:T], df; line=true)

Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_M, :sk_corr_half_M, :sk_corr_part_K, :ηk_corr_Γ, :ηk_corr_M, :ηk_corr_M2, :ηz)
groups = 3
groupsize = 32
nothing

In [ ]:
fig = Figure(size=(1200,800))
i = 11

scorrs = [:sk_corr_half_M, :sk_corr_M, :sk_corr_part_K]
ηcorrs = [:ηk_corr_M, :ηk_corr_Γ, :ηk_corr_M2]
for j in 1:groups
    fig[j,1] = ax1 = Axis(fig, title="$(scorrs[j])")
    fig[j,2] = ax2 = Axis(fig, title="$(ηcorrs[j])")
    df = mctimes[(j-1)*groupsize + i]
    lines!(ax1, df[:, scorrs[j]])
    lines!(ax2, getetapar.(df[:, ηcorrs[j]]))
end
println("(T, am) = ($(results.data[i,:T]), $(results.data[i,:am]))")
fig

In [ ]:
fig = Figure(size=(1200,800))
i = 9

scorrs = [:Energy, :Energy, :Energy]
ηcorrs = [:Energy, :Energy, :Energy]
for j in 1:groups
    fig[j,1] = ax1 = Axis(fig, title="$(scorrs[j])")
    fig[j,2] = ax2 = Axis(fig, title="$(ηcorrs[j])")
    df = mctimes[(j-1)*groupsize + i]
    lines!(ax1, df[:, scorrs[j]])
    lines!(ax2, df[:, ηcorrs[j]])
end
println("(T, am) = ($(results.data[i,:T]), $(results.data[i,:am]))")
fig

## AFM-FE to AFE Boundary

In [ ]:
results = JobResult("../jobs", "afmfe-afmafe")
df = results.data
for phase in [:stripe, :afm_afe, :afm_fe]
    df[:, Symbol(phase, "_auto")] = get_autocorr(results, Symbol("etak_corr_", phase))
end
Ts = unique(df[:,:T])
results

In [ ]:
fig = Figure(size=(1000, 300))

for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{Order} (T=%$T)", xlabel=L"a_m")
    data = subset(df, :T => x -> x .== T)
    generate_plot!(ax, :am, :etak_corr_afm_fe, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :etak_corr_afm_afe, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 300))

for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{Autocorr Time} $(T=%$T)$", xlabel=L"a_m")
    data = subset(df, :T => x -> x .== T)
    generate_plot!(ax, :am, :afm_fe_auto, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :afm_afe_auto, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(800, 400))

df = results.data
ax1 = fig[1,1] = Axis(fig, title=L"Energy Kurtosis vs. $a_m$", xlabel=L"a_m")
generate_plot!(ax1, :am, :energy_kurt, [:T], df; line=true)
ax2 = fig[1,2] = Axis(fig, title=L"Heat Capacity vs. $a_m$", xlabel=L"a_m")
generate_plot!(ax2, :am, :HeatCap, [:T], df; line=true)

Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
mctimes = get_mctime_data(results, :Energy)
groupsize = 10
nothing

In [ ]:
fig = Figure(size=(1200, 600))
axs = []
cols = 5

j = 2
min_i = 1
binsize = 0.001
as = subset(df, :T => T -> T .== Ts[j])[:, :am]
for i in min_i:length(as)
    L = Int(df[1, :Lx])
    a = round(as[i], sigdigits=4)
    fig[cld(i-min_i+1, cols), mod1(i-min_i+1, cols)] = ax = Axis(fig, title=L"T=%$(Ts[j]), a_m=%$a", xlabel="Energy", ylabel="Counts")
    Es = mctimes[i+groupsize*(j-1)][:, :Energy]
    minE = minimum(Es)
    maxE = maximum(Es)
    bins = minE:binsize:maxE
    hist!(ax, Es; bins, alpha=0.5)

    # β = df[j, :logL2fits][i]
    # μ1, μ2 = β[1], β[3]
    # σ1, σ2 = exp(β[2]), exp(β[4])
    # c = β[5]
    # norm = length(Es) * (maxE-minE) / (length(bins)-1)
    # p = (E -> begin
    #     ((1-c) * (2π*σ1^2)^(-1/2) * exp(-(E-μ1)^2 / (2σ1^2)) +
    #     c * (2π*σ2^2)^(-1/2) * exp(-(E-μ2)^2 / (2σ2^2))) * norm
    # end)
    # lines!(ax, minE..maxE, p)
    # p = E -> (1-c)*(2π*σ1^2)^(-1/2) * exp(-(E-μ1)^2 / (2σ1^2)) * norm
    # lines!(ax, minE..maxE, p)
    # p = E -> c*(2π*σ2^2)^(-1/2) * exp(-(E-μ2)^2 / (2σ2^2)) * norm
    # lines!(ax, minE..maxE, p)

    if length(axs) != 0
        # linkxaxes!(ax, axs[end])
        linkyaxes!(ax, axs[end])
    end
    push!(axs, ax)
end

fig

## Stripe to FM Boundary

In [ ]:
results = JobResult("../jobs", "stripe-fm")
df = results.data
df[:, :XiSpin] = corrlen.(df[:, :sr_corr])
df[:, :XiEta] = corrlen.(df[:,:etar_corr])
for phase in [:stripe, :afm_afe, :afm_fe]
    df[:, Symbol(phase, "_auto")] = get_autocorr(results, Symbol("etak_corr_", phase))
end
Ts = unique(df[:,:T])
results

In [ ]:
fig = Figure(size=(1000, 300))

for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{Order} (T=%$T)", xlabel=L"a_m")
    data = subset(df, :T => x -> x .== T)
    generate_plot!(ax, :am, :etak_corr_fm, data; line=true, label="fm")
    generate_plot!(ax, :am, :etak_corr_stripe, data; line=true, label="stripe")
end

Legend(fig[:,4], ax, merge=true)
fig

In [ ]:
fig = Figure(size=(800, 400))

df = results.data
ax1 = fig[1,1] = Axis(fig, title="Energy vs. am", xlabel=L"a_m")
generate_plot!(ax1, :am, :XiSpin, [:T], df; line=true)
ax2 = fig[1,2] = Axis(fig, title="Heat Capacity vs. am", xlabel=L"a_m")
generate_plot!(ax2, :am, :XiEta, [:T], df; line=true)

Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
fig = Figure(size=(800, 400))

df = results.data
ax1 = fig[1,1] = Axis(fig, title=L"Energy Kurtosis vs. $a_m$", xlabel=L"a_m")
generate_plot!(ax1, :am, :energy_kurt, [:T], df; line=true)
ax2 = fig[1,2] = Axis(fig, title=L"Heat Capacity vs. $a_m$", xlabel=L"a_m")
generate_plot!(ax2, :am, :HeatCap, [:T], df; line=true)

Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 300))

for i in eachindex(Ts)
    T = Ts[i]
    global ax = fig[cld(i, 3), mod1(i, 3)] = Axis(fig, title=L"$\eta$ \textbf{Autocorr Time} $(T=%$T)$", xlabel=L"a_m")
    data = subset(df, :T => x -> x .== T)
    generate_plot!(ax, :am, :afm_fe_auto, data; line=true, label="afm_fe")
    generate_plot!(ax, :am, :afm_afe_auto, data; line=true, label="afm_afe")
end

Legend(fig[:,4], ax, merge=true)
fig

## Full Phase Diagram

In [ ]:
results = JobResult("../jobs", "full-diagram")
sort!(results.data, [:T])
ams = sort(unique(results.data[:, :am]))
ers = sort(unique(results.data[:, :er]))
params = Iterators.product(ams, ers)
Ts = unique(results.data[:, :T])
Ls = unique(results.data[:, :Lx])
phases = [:stripe, :fm, :afm_fe, :afm_afe]
thresh = 0.1

results.data[:, :energy_auto] = get_autocorr(results, :Energy)
for phase in [:stripe, :fm, :afm_fe, :afm_afe]
    corrname = Symbol(:etak_corr_, phase)
    results.data[:, Symbol(corrname, :_auto)] = get_autocorr(results, corrname)
end

results

In [ ]:
vars = [Symbol(:etak_corr_, phase) for phase in phases]
df_extrap = combine(
    groupby(results.data, [:am, :er, :T]),
    [[:Lx, x] => extrapolate => x for x in vars]...,
    [[:Lx, x] => pearson => Symbol(x, "_pearson") for x in vars]...
)

In [ ]:
cols = 2
fig = Figure(size=(600,600))
for i in 1:4
    phase = getindex(phases, i)
    name = phase == :stripe ? "Stripe" : uppercase(replace(String(phase), '_' => '-'))
    pos = [1 + fld(i-1, cols), 1 + 2 * rem(i-1, cols)]
    fig[pos...] = ax = Axis(fig, title=L"%$(name) $T_c$ (K)", xlabel = L"a_m", ylabel=L"\epsilon_r")

    corrname = Symbol(:etak_corr_, phase)
    corrs = broadcast(((am, er),) -> begin
        data = selectamer(df_extrap, am, er)
        Tc = estTc(data[:, :T], data[:, corrname]; thresh)
        H0 = getH0(am, er)
        return H0 / kB * try
            Tc.val
        catch
            Tc
        end
    end, params)
    hm = heatmap!(ax, ams, ers, corrs)
    Colorbar(fig[pos + [0,1]...], hm)
end
save("plots/full_diagram_Tc.png", fig)
fig

In [ ]:
fig = Figure(size=(width, width), figure_padding=1, fontsize=9pt)
cols = 2

ax = Axis(
    fig[1,1], title=L"Critical $T$",
    xlabel = L"a_m", ylabel=L"\varepsilon_r",
    xlabelpadding=0, ylabelpadding=0,
    xticksize=2, yticksize=2,
    xlabelvisible=false,
    xticksvisible=false, xticklabelsvisible=false
)

Tcs = broadcast(((am, er),) -> begin
    data = selectamer(df_extrap, am, er)
    maxTc = 0.0
    for phase in phases
        corrname = Symbol(:etak_corr_, phase)
        Tc = estTc(data[:, :T], data[:, corrname]; thresh)
        H0 = getH0(am, er)
        maxTc = max(maxTc, H0 / kB * try
            Tc.val
        catch
            Tc
        end)
    end
    return maxTc == 0 ? -1 : maxTc
end, params)
hm = heatmap!(
    ax, ams, ers, Tcs,
    lowclip=:snow3, colorrange=(0, maximum(Tcs)),
)
Colorbar(
    fig[1,2], hm,
    label=L"T_c\ (K)",
    halign=:left,
    ticks=[0, 30, 60],
    ticksize=2,
)

ax = Axis(fig[2,1],
    xlabel = L"a_m", ylabel=L"\varepsilon_r",
    xlabelpadding=0, ylabelpadding=0,
    xticksize=2, yticksize=2,
    xticks = [5, 8, 11]
)
phasetonum = Dict(:none => NaN, :stripe => 1, :fm => 2, :afm_fe => 3, :afm_afe => 4)
dom_phases = broadcast(((am, er),) -> begin
    data = subset(selectamer(df_extrap, am, er))
    ground = [:none, 0.0]
    for phase in phases
        corrname = Symbol(:etak_corr_, phase)
        corr = maximum(data[:, corrname])
        if corr > thresh && corr > ground[2]
            ground[1] = phase
            ground[2] = corr
        end
    end
    return phasetonum[ground[1]]
end, params)
colormap = [Makie.wong_colors()[i] for i in 1:4]
hm = heatmap!(ax, ams, ers, dom_phases; colormap, nan_color=:snow3)

Legend(fig[2,2],
    [PolyElement(; color) for color in colormap[1:4]],
    ["Stripe", "FM", "AFM-FE", "AFM-AFE"],
    halign=:left,
    tellwidth=true,
    patchsize=(5,5),
)

rowgap!(fig.layout, 5)
colgap!(fig.layout, 5)
save("plots/full_diagram_combTc.pdf", fig)
fig

In [ ]:
cols = 3
fig = Figure(size=(1.5width, 0.7width), figure_padding=1, fontsize=9pt)

for i in 1:3
    coupl = getindex([:J_EzEz, :J_EzEz_SS, :J_SS], i)
    pos = [2*cld(i,cols)-1, mod1(i,cols)]
    ax = Axis(
        fig[pos...],
        title="$coupl",
        xlabel = L"a_m", ylabel=L"\varepsilon_r",
        xlabelpadding=0, ylabelpadding=0,
        xticksize=2, yticksize=2,
        xticks = [5, 8, 11]
    )

    Js = getfield.(WignerParams.("params_interpolated_quarter.jld2", params), coupl)
    hm = heatmap!(ax, ams, ers, real.(Js))
    Colorbar(fig[pos+[1,0]...], hm, vertical=false, flipaxis=false)
end

rowgap!(fig.layout, 5)
colgap!(fig.layout, 5)
fig

In [ ]:
fig = Figure(size=(width, width), figure_padding=1, fontsize=9pt)

L = Int(Ls[end])
Ts = unique(results.data[:, :T])
i = 3
ax = Axis(
    fig[1,1], title=L"Heat Capacity ($%$L\times %$L$ Lattice, $T=%$(Ts[i])H_0$)",
    xlabel = L"a_m", ylabel=L"\varepsilon_r",
    xlabelpadding=0, ylabelpadding=0,
    xticksize=2, yticksize=2,
    xticks = [5, 8, 11]
)
df = subset(results.data, :Lx => Lx -> Lx .== L, :T => T -> T .== Ts[i])
Tcs = broadcast(((am, er),) -> begin
    data = selectamer(df, am, er)
    return data[1, :HeatCap].val
end, params)
hm = heatmap!(ax, ams, ers, Tcs, colorscale=log10)
Colorbar(fig[1,2], hm, label=L"c", ticksize=2)

fig

In [ ]:
fig = Figure(size=(width, width), figure_padding=1, fontsize=9pt)

L = Int(Ls[end])
Ts = unique(results.data[:, :T])
i = 2
ax = Axis(
    fig[1,1], title=L"Energy Autocorr ($T=%$(Ts[i])H_0$)",
    xlabel = L"a_m", ylabel=L"\varepsilon_r",
    xlabelpadding=0, ylabelpadding=0,
    xticksize=2, yticksize=2,
    xticks = [5, 8, 11]
)
df = subset(results.data, :Lx => Lx -> Lx .== L, :T => T -> T .== Ts[i])
Tcs = broadcast(((am, er),) -> begin
    data = selectamer(df, am, er)
    return data[1, :energy_auto]
end, params)
hm = heatmap!(ax, ams, ers, Tcs)
Colorbar(fig[1,2], hm, label=L"c", ticksize=2)

fig

In [ ]:
cols = 2
j = 2
fig = Figure(size=(600,600))
for i in 1:4
    phase = getindex([:stripe, :fm, :afm_fe, :afm_afe], i)
    name = phase == :stripe ? "Stripe" : uppercase(replace(String(phase), '_' => '-'))
    pos = [1 + fld(i-1, cols), 1 + 2 * rem(i-1, cols)]
    fig[pos...] = ax = Axis(
        fig, title="$(name) Autocorr",
        xlabel = L"a_m", ylabel=L"\epsilon_r",
        xticks=[5, 8, 11]
    )

    corrname = Symbol(:etak_corr_, phase, :_auto)
    df = subset(results.data, :Lx => Lx -> Lx .== 84)
    corrs = broadcast(((am, er),) -> begin
        data = selectamer(df, am, er)
        return data[j, corrname]
    end, params)
    hm = heatmap!(ax, ams, ers, corrs)
    Colorbar(fig[pos + [0,1]...], hm)
end

fig

In [ ]:
cols = 3
for phase in [:stripe, :fm, :afm_fe, :afm_afe]
    fig = Figure(size=(1000,600))
    name = phase == :stripe ? "Stripe" : uppercase(replace(String(phase), '_' => '-'))
    for i in 1:length(Ts)
        pos = [1 + fld(i-1, cols), 1 + 2 * rem(i-1, cols)]
        setT = Ts[i]
        fig[pos...] = ax = Axis(fig, title=L"\textbf{%$(name) Eta Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
        df = subset(results.data, :T => T -> T .== setT)
        corrname = Symbol(:etak_corr_, phase)
        corrs = broadcast(((am, er),) -> selectamer(df, am, er)[1, corrname].val, params)
        hm = heatmap!(ax, ams, ers, corrs)
        Colorbar(fig[pos + [0,1]...], hm)
    end
    # display(fig)
    save("plots/full_diagram_eta_$phase.pdf", fig)
end

In [ ]:
cols = 3
for phase in [:stripe, :fm, :afm_fe, :afm_afe]
    fig = Figure(size=(1000,600))
    name = phase == :stripe ? "Stripe" : uppercase(replace(String(phase), '_' => '-'))
    for i in 1:length(Ts)
        pos = [1 + fld(i-1, cols), 1 + 2 * rem(i-1, cols)]
        setT = Ts[i]
        fig[pos...] = ax = Axis(fig, title=L"\textbf{%$(name) Eta Error} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
        df = subset(results.data, :T => T -> T .== setT)
        corrname = Symbol(:etak_corr_, phase)
        corrs = broadcast(((am, er),) -> selectamer(df, am, er)[1, corrname].err, params)
        hm = heatmap!(ax, ams, ers, corrs)
        Colorbar(fig[pos + [0,1]...], hm)
    end
    # display(fig)
    save("plots/full_diagram_etaerr_$phase.pdf", fig)
end

In [ ]:
cols = 3
for phase in [:half_M, :Γ, :M, :part_K]
    fig = Figure(size=(1000,600))
    name = phase == :stripe ? "Stripe" : uppercase(replace(String(phase), '_' => '-'))
    for i in 1:length(Ts)
        pos = [1 + fld(i-1, cols), 1 + 2 * rem(i-1, cols)]
        setT = Ts[i]
        fig[pos...] = ax = Axis(fig, title=L"\textbf{%$(name) Eta Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
        df = subset(results.data, :T => T -> T .== setT)
        corrname = Symbol(:sk_corr_, phase)
        corrs = broadcast(((am, er),) -> selectamer(df, am, er)[1, corrname].val, params)
        hm = heatmap!(ax, ams, ers, corrs)
        Colorbar(fig[pos + [0,1]...], hm)
    end
    # display(fig)
    save("plots/full_diagram_spin_$phase.pdf", fig)
end

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_near_Γ, :ηk_corr_near_Γ,
    :sk_corr_near_M, :sk_corr_near_M2, :sk_corr_near_M3,
    :sk_corr_near_half_M, :sk_corr_near_half_M2, :sk_corr_near_half_M3,
    :sk_corr_near_part_K, :sk_corr_near_part_K2, :sk_corr_near_part_K3,
)
groups = length(Ts)
groupsize = length(ams) * length(ers)
nothing

In [ ]:
i = 46
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_M] + df[:, :sk_corr_near_M2] + df[:, :sk_corr_near_M3]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin M Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig

In [ ]:
i = 12
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_Γ]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin} $\Gamma$ \textbf{Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig

In [ ]:
i = 13
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_half_M] + df[:, :sk_corr_near_half_M2] + df[:, :sk_corr_near_half_M3]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin M/2 Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig

In [ ]:
i = 21
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_part_K] + df[:, :sk_corr_near_part_K2] + df[:, :sk_corr_near_part_K3]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin 3K/4 Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig